# Gallbladder Disease Detection — Explainable CNN+ViT Fusion
### Ultrasound classification (9 classes) — Colab training notebook

Pipeline: **Data loading → Augmentation → CNN+ViT fusion model → Training → Evaluation → Grad-CAM → Export for local inference**

Dataset: [Gallblader Diseases Dataset, Mendeley Data v2](https://data.mendeley.com/datasets/r6h24d2d3y/2) — 10,692 ultrasound images, 9 classes, ~1200 images/class.

**Note on scope:** this dataset is ultrasound-only. The model/architecture below is written to be modality-agnostic (an extra CT/MRCP encoder branch can be added later using the same fusion pattern used here for CNN+ViT).


## 0. Setup — installs, imports, config, seed


In [ ]:
# Run once per Colab session
!pip install -q timm grad-cam torchmetrics albumentations onnx onnxruntime


In [ ]:
import os, json, random, shutil, zipfile, glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              confusion_matrix, roc_auc_score, roc_curve)
from sklearn.preprocessing import label_binarize

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

CONFIG = {
    'data_root': '/content/gallbladder_dataset',   # folder containing 9 class subfolders after unzip
    'work_dir':  '/content/work',                    # checkpoints, logs, exported artifacts
    'img_size': 224,
    'batch_size': 32,
    'num_workers': 2,
    'epochs': 30,
    'lr': 3e-4,
    'weight_decay': 1e-4,
    'early_stop_patience': 6,
    'val_size': 0.15,
    'test_size': 0.15,
    'cnn_backbone': 'efficientnet_b0',
    'vit_backbone': 'vit_small_patch16_224',
    'fusion_hidden': 256,
    'label_smoothing': 0.1,
}
os.makedirs(CONFIG['work_dir'], exist_ok=True)
print(json.dumps(CONFIG, indent=2))


## 1. Data acquisition & organization

Two ways to get the data into Colab:
1. Download the zip manually from Mendeley, upload it to your Google Drive, then mount Drive and unzip.
2. Upload the zip directly via the Colab file picker (slower for large files, fine for testing).

After unzip, `CONFIG['data_root']` should contain **one subfolder per class**, each full of image files — that's what the loader below expects. Adjust `class_names` if the extracted folder names differ.


In [ ]:
# Option A: Google Drive (recommended)
from google.colab import drive
drive.mount('/content/drive')

# EDIT this path to wherever you uploaded the dataset zip in your Drive
DRIVE_ZIP_PATH = '/content/drive/MyDrive/datasets/gallbladder_diseases_v2.zip'

if os.path.exists(DRIVE_ZIP_PATH) and not os.path.exists(CONFIG['data_root']):
    os.makedirs(CONFIG['data_root'], exist_ok=True)
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
        zf.extractall(CONFIG['data_root'])
    print('Extracted to', CONFIG['data_root'])
else:
    print('Zip not found at DRIVE_ZIP_PATH or data_root already populated — edit the path above, '
          'or use the Option B upload cell instead.')


In [ ]:
# Option B: manual upload (uncomment to use instead of Drive)
# from google.colab import files
# uploaded = files.upload()  # select the zip in the picker
# zip_name = list(uploaded.keys())[0]
# os.makedirs(CONFIG['data_root'], exist_ok=True)
# with zipfile.ZipFile(zip_name, 'r') as zf:
#     zf.extractall(CONFIG['data_root'])


In [ ]:
# Discover class folders and build a (filepath, label) index.
# Handles the case where the zip nests everything one level deeper (common with Mendeley exports) —
# it walks down until it finds folders that actually contain images.

IMG_EXT = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

def find_class_dirs(root):
    root = Path(root)
    candidates = [p for p in root.rglob('*') if p.is_dir()]
    class_dirs = []
    for d in candidates:
        imgs = [f for f in d.iterdir() if f.suffix.lower() in IMG_EXT]
        if len(imgs) >= 20:   # heuristic: a real class folder, not an empty/nested wrapper
            class_dirs.append(d)
    return sorted(class_dirs)

class_dirs = find_class_dirs(CONFIG['data_root'])
print(f'Found {len(class_dirs)} class folders:')
for d in class_dirs:
    n = len([f for f in d.iterdir() if f.suffix.lower() in IMG_EXT])
    print(f'  {d.name:45s} {n:5d} images')


In [ ]:
# Build the full file index
records = []
for d in class_dirs:
    label = d.name
    for f in d.iterdir():
        if f.suffix.lower() in IMG_EXT:
            records.append({'filepath': str(f), 'label': label})

df = pd.DataFrame(records)
class_names = sorted(df['label'].unique().tolist())
label2idx = {c: i for i, c in enumerate(class_names)}
idx2label = {i: c for c, i in label2idx.items()}
df['label_idx'] = df['label'].map(label2idx)

print(df['label'].value_counts())
print('\nTotal images:', len(df))


**Patient-leakage caution:** if the original filenames encode a patient/study ID (check a sample of filenames — Mendeley radiology sets often do), split by that ID rather than by image, otherwise frames from the same study can leak between train/test and inflate your reported accuracy. The stratified split below is image-level by default; switch to `GroupShuffleSplit` on a parsed patient ID if you find one.


In [ ]:
# Stratified train/val/test split (image-level — see note above for patient-level splitting)
train_df, temp_df = train_test_split(
    df, test_size=CONFIG['val_size'] + CONFIG['test_size'],
    stratify=df['label_idx'], random_state=SEED)

relative_test = CONFIG['test_size'] / (CONFIG['val_size'] + CONFIG['test_size'])
val_df, test_df = train_test_split(
    temp_df, test_size=relative_test,
    stratify=temp_df['label_idx'], random_state=SEED)

print('Train:', len(train_df), 'Val:', len(val_df), 'Test:', len(test_df))

# Class weights for the loss (helps if any class ends up imbalanced after inspection)
counts = train_df['label_idx'].value_counts().sort_index().values
class_weights = (counts.sum() / (len(counts) * counts)).astype('float32')
class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to(device)
print('Class weights:', dict(zip(class_names, class_weights.round(2))))


## 2. Augmentation & Dataset/DataLoader

Augmentations are kept **ultrasound-appropriate**: no aggressive color jitter (US is grayscale/near-grayscale and color shifts are meaningless), no strong elastic warps that would distort anatomical shape beyond realism, and no vertical flips of hard-to-justify orientation for some probe angles — adjust if your images are consistently oriented.


In [ ]:
IMG_SIZE = CONFIG['img_size']
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tfms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=10, p=0.5, border_mode=cv2.BORDER_CONSTANT),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.GaussNoise(var_limit=(5.0, 20.0), p=0.3),          # mild speckle-like noise robustness
    A.CLAHE(clip_limit=2.0, p=0.2),                        # helps with US contrast variation
    A.CoarseDropout(max_holes=4, max_height=16, max_width=16, p=0.2),  # occlusion robustness
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

eval_tfms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])


In [ ]:
class GallbladderDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['filepath'], cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        augmented = self.transform(image=img)
        img_t = augmented['image']
        label = int(row['label_idx'])
        return img_t, label

train_ds = GallbladderDataset(train_df, train_tfms)
val_ds   = GallbladderDataset(val_df, eval_tfms)
test_ds  = GallbladderDataset(test_df, eval_tfms)

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True,
                           num_workers=CONFIG['num_workers'], pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False,
                           num_workers=CONFIG['num_workers'], pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False,
                           num_workers=CONFIG['num_workers'], pin_memory=True)

print('Batches — train:', len(train_loader), 'val:', len(val_loader), 'test:', len(test_loader))


In [ ]:
# Sanity check: visualize a batch
imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
std  = torch.tensor(IMAGENET_STD).view(3,1,1)
for i, ax in enumerate(axes.flat):
    img = imgs[i] * std + mean
    ax.imshow(img.permute(1,2,0).clamp(0,1).numpy())
    ax.set_title(idx2label[labels[i].item()], fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()


## 3. Model — CNN + ViT feature fusion

Two pretrained backbones (EfficientNet-B0 for local texture/edge features, ViT-Small for global/contextual attention) each produce an embedding; embeddings are concatenated and passed through a small fusion head. This is the practical way to realize a 'multimodal-style' architecture on a single-modality dataset — two complementary feature views of the same ultrasound image — and it keeps Grad-CAM straightforward on the CNN branch while the ViT branch adds attention-based context.


In [ ]:
class CNNViTFusion(nn.Module):
    def __init__(self, num_classes, cnn_name=CONFIG['cnn_backbone'], vit_name=CONFIG['vit_backbone'],
                 fusion_hidden=CONFIG['fusion_hidden'], dropout=0.3):
        super().__init__()
        # feature extractors, no classification head (num_classes=0 -> pooled embedding output)
        self.cnn = timm.create_model(cnn_name, pretrained=True, num_classes=0, global_pool='avg')
        self.vit = timm.create_model(vit_name, pretrained=True, num_classes=0)

        cnn_dim = self.cnn.num_features
        vit_dim = self.vit.num_features

        self.fusion = nn.Sequential(
            nn.Linear(cnn_dim + vit_dim, fusion_hidden),
            nn.BatchNorm1d(fusion_hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(fusion_hidden, num_classes)

    def forward(self, x, return_embeddings=False):
        cnn_feat = self.cnn(x)          # (B, cnn_dim)
        vit_feat = self.vit(x)          # (B, vit_dim)
        fused = torch.cat([cnn_feat, vit_feat], dim=1)
        fused = self.fusion(fused)
        logits = self.classifier(fused)
        if return_embeddings:
            return logits, fused
        return logits

num_classes = len(class_names)
model = CNNViTFusion(num_classes=num_classes).to(device)
print(f'CNN dim: {model.cnn.num_features}, ViT dim: {model.vit.num_features}, classes: {num_classes}')
total_params = sum(p.numel() for p in model.parameters())
print(f'Total params: {total_params/1e6:.1f}M')


## 4. Training loop — loss, optimizer, scheduler, early stopping, checkpointing


In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights_t, label_smoothing=CONFIG['label_smoothing'])
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

best_ckpt_path = os.path.join(CONFIG['work_dir'], 'best_model.pt')


In [ ]:
def run_epoch(loader, train_mode):
    model.train() if train_mode else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train_mode):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train_mode:
                optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                logits = model(imgs)
                loss = criterion(logits, labels)
            if train_mode:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * imgs.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)
    return total_loss / total, correct / total


In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
patience_counter = 0

for epoch in range(1, CONFIG['epochs'] + 1):
    train_loss, train_acc = run_epoch(train_loader, train_mode=True)
    val_loss, val_acc = run_epoch(val_loader, train_mode=False)
    scheduler.step()

    history['train_loss'].append(train_loss); history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss); history['val_acc'].append(val_acc)

    print(f"Epoch {epoch:02d}/{CONFIG['epochs']} | "
          f"train_loss {train_loss:.4f} acc {train_acc:.4f} | "
          f"val_loss {val_loss:.4f} acc {val_acc:.4f} | lr {scheduler.get_last_lr()[0]:.2e}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'config': CONFIG,
            'class_names': class_names,
            'epoch': epoch,
            'val_acc': val_acc,
        }, best_ckpt_path)
        print(f'  -> saved new best checkpoint (val_acc={val_acc:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['early_stop_patience']:
            print(f'Early stopping at epoch {epoch} (no improvement for {patience_counter} epochs)')
            break


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='train'); axes[0].plot(history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
axes[1].plot(history['train_acc'], label='train'); axes[1].plot(history['val_acc'], label='val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend()
plt.tight_layout(); plt.savefig(os.path.join(CONFIG['work_dir'], 'training_curves.png'), dpi=150)
plt.show()


## 5. Evaluation on held-out test set

Full metric suite: accuracy, sensitivity (recall), specificity, precision, F1 — per class and macro-averaged — plus one-vs-rest AUC and a confusion matrix.


In [ ]:
# Load best checkpoint
ckpt = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded best checkpoint from epoch {ckpt['epoch']} (val_acc={ckpt['val_acc']:.4f})")


In [ ]:
all_probs, all_preds, all_labels = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        probs = F.softmax(logits, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
        all_probs.append(probs)
        all_preds.append(preds)
        all_labels.append(labels.numpy())

all_probs = np.concatenate(all_probs)
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, support = precision_recall_fscore_support(
    all_labels, all_preds, labels=range(num_classes), zero_division=0)

# Specificity per class: TN / (TN + FP) from the confusion matrix
cm = confusion_matrix(all_labels, all_preds, labels=range(num_classes))
specificity = []
for i in range(num_classes):
    tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
    fp = cm[:, i].sum() - cm[i, i]
    specificity.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)

y_bin = label_binarize(all_labels, classes=range(num_classes))
auc_per_class = []
for i in range(num_classes):
    try:
        auc_per_class.append(roc_auc_score(y_bin[:, i], all_probs[:, i]))
    except ValueError:
        auc_per_class.append(float('nan'))

results_df = pd.DataFrame({
    'class': class_names,
    'support': support,
    'precision': precision,
    'sensitivity_recall': recall,
    'specificity': specificity,
    'f1': f1,
    'auc': auc_per_class,
})
print(f'Overall test accuracy: {acc:.4f}')
print(f"Macro AUC: {np.nanmean(auc_per_class):.4f}")
results_df.round(4)


In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(num_classes)); ax.set_xticklabels(class_names, rotation=90)
ax.set_yticks(range(num_classes)); ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Confusion Matrix (test set)')
for i in range(num_classes):
    for j in range(num_classes):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=8)
plt.colorbar(im)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['work_dir'], 'confusion_matrix.png'), dpi=150)
plt.show()

results_df.to_csv(os.path.join(CONFIG['work_dir'], 'test_metrics.csv'), index=False)


## 6. Explainability — Grad-CAM

Grad-CAM is applied to the **CNN branch** (last conv block of EfficientNet), which gives clean spatial heatmaps. For the ViT branch, attention-rollout is the analogous technique (not implemented here to keep the notebook focused — flag it in your report as a stated extension, or ask for it as a follow-up).


In [ ]:
# Locate EfficientNet's last conv layer for Grad-CAM
target_layers = [model.cnn.conv_head]   # timm EfficientNet exposes this attribute

def gradcam_for_image(img_tensor, true_label_idx, pred_label_idx=None):
    """img_tensor: normalized (3,H,W) tensor (no batch dim)."""
    input_tensor = img_tensor.unsqueeze(0).to(device)
    target_class = pred_label_idx if pred_label_idx is not None else true_label_idx
    targets = [ClassifierOutputTarget(target_class)]

    with GradCAM(model=model, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

    # de-normalize for visualization
    img_vis = img_tensor.cpu() * std + mean
    img_vis = img_vis.permute(1, 2, 0).clamp(0, 1).numpy()
    overlay = show_cam_on_image(img_vis, grayscale_cam, use_rgb=True)
    return img_vis, overlay


In [ ]:
# Sample a few test images per class-of-interest and render Grad-CAM overlays
model.eval()
n_samples = 6
sample_idx = np.random.choice(len(test_ds), n_samples, replace=False)

fig, axes = plt.subplots(2, n_samples, figsize=(3*n_samples, 6))
for col, idx in enumerate(sample_idx):
    img_t, label = test_ds[idx]
    with torch.no_grad():
        logits = model(img_t.unsqueeze(0).to(device))
        pred = logits.argmax(dim=1).item()
    img_vis, overlay = gradcam_for_image(img_t, label, pred)
    axes[0, col].imshow(img_vis); axes[0, col].axis('off')
    axes[0, col].set_title(f'True: {idx2label[label]}', fontsize=8)
    axes[1, col].imshow(overlay); axes[1, col].axis('off')
    correct = '✓' if pred == label else '✗'
    axes[1, col].set_title(f'Pred: {idx2label[pred]} {correct}', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['work_dir'], 'gradcam_samples.png'), dpi=150)
plt.show()


## 7. Export for local inference

**Do not use `.pkl` for this deep model** — pickle is fragile across library versions and not the standard for PyTorch. Instead we save:
- `model_weights.pt` — `state_dict` (load into the same `CNNViTFusion` class definition locally)
- `model.onnx` — framework-agnostic export, usable with `onnxruntime` without needing PyTorch/timm installed locally
- `class_mapping.json` — index-to-label mapping (easy to forget, breaks predictions silently if lost)
- `preprocessing_config.json` — image size + normalization stats, so local inference matches training exactly

Zip everything and download, or push to Drive.


In [ ]:
export_dir = os.path.join(CONFIG['work_dir'], 'export')
os.makedirs(export_dir, exist_ok=True)

# 1) PyTorch state_dict (for loading back into this same model class in another PyTorch env)
torch.save(model.state_dict(), os.path.join(export_dir, 'model_weights.pt'))

# 2) ONNX export (best for lightweight local inference, no timm/torch dependency needed to serve)
model.eval()
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
onnx_path = os.path.join(export_dir, 'model.onnx')
torch.onnx.export(
    model, dummy_input, onnx_path,
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
)
print('Exported ONNX model to', onnx_path)

# quick sanity check that ONNX and PyTorch agree
import onnxruntime as ort
ort_session = ort.InferenceSession(onnx_path)
onnx_out = ort_session.run(None, {'input': dummy_input.cpu().numpy()})[0]
with torch.no_grad():
    torch_out = model(dummy_input).cpu().numpy()
print('Max abs diff PyTorch vs ONNX:', np.abs(onnx_out - torch_out).max())


In [ ]:
# 3) class mapping + preprocessing config (critical — don't skip)
with open(os.path.join(export_dir, 'class_mapping.json'), 'w') as f:
    json.dump({'idx2label': idx2label, 'label2idx': label2idx}, f, indent=2)

preprocessing_config = {
    'img_size': IMG_SIZE,
    'normalize_mean': IMAGENET_MEAN,
    'normalize_std': IMAGENET_STD,
    'color_mode': 'RGB',
    'cnn_backbone': CONFIG['cnn_backbone'],
    'vit_backbone': CONFIG['vit_backbone'],
}
with open(os.path.join(export_dir, 'preprocessing_config.json'), 'w') as f:
    json.dump(preprocessing_config, f, indent=2)

# also copy the evaluation artifacts for the report
for fname in ['training_curves.png', 'confusion_matrix.png', 'gradcam_samples.png', 'test_metrics.csv']:
    src = os.path.join(CONFIG['work_dir'], fname)
    if os.path.exists(src):
        shutil.copy(src, export_dir)

print('Export folder contents:')
for f in sorted(os.listdir(export_dir)):
    print(' -', f)


In [ ]:
# 4) Zip and download
zip_path = '/content/gallbladder_model_export.zip'
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', export_dir)

from google.colab import files
files.download(zip_path)


---
### Local inference — how to load these artifacts

```python
import torch, json
from your_model_file import CNNViTFusion   # paste the class definition from Section 3

with open('class_mapping.json') as f:
    mapping = json.load(f)
idx2label = {int(k): v for k, v in mapping['idx2label'].items()}

model = CNNViTFusion(num_classes=len(idx2label))
model.load_state_dict(torch.load('model_weights.pt', map_location='cpu'))
model.eval()
```

Or, for a lighter local footprint without PyTorch/timm, use `onnxruntime` with `model.onnx` and replicate the exact preprocessing in `preprocessing_config.json` (resize → RGB → normalize with the given mean/std → CHW tensor).

**Pin your versions.** Before leaving Colab, run `!pip freeze | grep -E "torch|timm|onnx"` and copy those exact versions into a local `requirements.txt` — mismatched versions are the most common cause of 'works in Colab, breaks locally' errors.
